# Training a translation LoRA, step by step

**What this does, in one line:** you give it sentences in one language with their translations, and
it teaches an existing translation model to translate more like your examples. The result is a
**LoRA**: a small file of adjustments that sits on top of the model, the way a lens sits in front of
a camera. The model itself never changes.

The whole job, start to finish:

```
your sentence pairs  +  a model  ──►  train  ──►  a LoRA  ──►  check it helped  ──►  use it in volis
```

Every step below is one cell. Run them in order with **Shift+Enter**. Each cell says what it did,
and stops with a plain message if something needs fixing. You don't need to understand every line
of code; the text above each cell says what it's for.

**Out of the box it runs on its own practice data,** 40 English → Mexican Spanish sentences, so you
can try the whole thing once before using your own. The practice run teaches the model almost
nothing (40 sentences is far too few). It's there to show every step working.

**How to open this notebook:** the same way as `attach-a-lora.ipynb`. In VS Code, open the volis
folder, open this file, click **Select Kernel → Python Environments → .venv**, and say yes if it
offers to install `ipykernel`.

## Part 1: what you need before starting

This is the part that's easy to get lost in. There are only two things to get right: **the model**
and **the data**.

### The model: it has to be the "original" version

Models are published in several file formats. They're the same model, packed differently for
different jobs. Training needs the original, unpacked one.

| You see files ending in | What it is | Can you train it? |
|---|---|---|
| `.safetensors`, with `config.json` and `tokenizer.json` beside it | The original, as its makers published it on Hugging Face | **Yes. This is what you want.** |
| `.bin`, `.pth`, `.pt` | The same thing in PyTorch's older format | Yes, but avoid it: opening these files can run code hidden inside them. `.safetensors` files can't. |
| `.gguf` | Packed small and fast, for *running* (volis, llama.cpp, LM Studio) | **No.** It's the finished product, not the raw material. |
| `.onnx` | Packed for running on other engines | **No.** Same reason. |

So you train on the `.safetensors` version, and volis runs the `.gguf` version. That means you
need **the same model in both forms**. A LoRA fits only the exact model it was trained on, like a
key cut for one lock (`attach-a-lora.ipynb`, Step 2). On Hugging Face, the original is usually
the page named after the model (`Qwen/Qwen3-0.6B`), and the GGUF is a separate page with `GGUF` in
its name.

**Which model?** This notebook starts with **Qwen3-0.6B**: small enough to train on almost any
NVIDIA graphics card in minutes. Once the steps make sense, the same notebook trains a bigger,
better translator. The bigger the model, the more graphics memory it needs:

| Model | Graphics memory to train it | Good for |
|---|---|---|
| Qwen3-0.6B | about 6 GB (a laptop GPU is fine) | learning the steps |
| Qwen3-4B, Gemma 4 E4B | about 24 GB or more (RTX 5090, A100) | real translation work |

Without an NVIDIA card it still runs, on the processor, but slowly: fine for the practice data,
too slow for real data.

### The data: sentence pairs

What you train with decides what kind of model you can improve:

| You have | It trains | In this notebook? |
|---|---|---|
| **Sentences and their translations** | a **translator** | **Yes** |
| Words and their translations (a word list) | nothing, on its own. Single words don't show a model how sentences are put together, and a model trained on word lists starts answering in word lists. Use them as a **glossary** in volis instead, or put each word into a sentence first. | No |
| Recordings with a written transcript of each | a **recognizer** (speech to text, like Whisper) | No: that's a different job, `model-converter`'s Whisper training |
| Recordings with recorded translations | nothing volis can use yet | No |

**The file:** a spreadsheet saved as **CSV**, two columns, one pair per row. The first row names
the columns `source` and `target`:

```
source,target
"Where is the bus stop?","¿Dónde está la parada del camión?"
"I'll call you later.","Al rato te marco."
```

Save it as **CSV UTF-8** (Excel: *File → Save As → CSV UTF-8*). That keeps accents and non-Latin
letters intact. A tab-separated `.tsv`, or a `.jsonl` file with `source` and `target` on each line,
works too.

**How many pairs?**
- about 200: you can start to see a difference;
- 1,000 to 5,000: a real improvement;
- more helps, but **good pairs matter more than many pairs**. 1,000 careful pairs beat 10,000
  sloppy ones.

**What makes a good pair:**
- real, everyday sentences, the way people actually say them, in the dialect you want;
- a natural translation, not word for word;
- the same spelling habits throughout;
- one sentence or one short exchange per row, not whole paragraphs.

The notebook checks the file for you (empty rows, duplicates, rows where one side is far longer
than the other) and says what it dropped and why.

## Part 2: your choices

Everything you'd change lives in this one cell. To begin, leave it as it is: it uses the practice
data and the small practice model.

- `MODEL`: the model to train, by its name on Hugging Face (the part after `huggingface.co/`).
  It's kept in `training\base\` inside the volis folder.
- `DATA_FILE`: your CSV. Empty means the practice pairs.
- `SOURCE`, `TARGET`: the language of each column, as volis names them: `en`, `es`, `es-MX`,
  `ar`, `ar-IQ`, `fa`, and so on. A tag with a region (`es-MX`, `ar-IQ`) tells the model to use
  that dialect, exactly as volis does.
- `RUN_NAME`: a name for this attempt. Everything it makes goes in a folder with this name.

In [ ]:
MODEL      = "Qwen/Qwen3-0.6B"                # its name on Hugging Face
DATA_FILE  = r""                              # e.g. r"D:\my-data\english-mexican.csv"
SOURCE     = "en"
TARGET     = "es-MX"
RUN_NAME   = "practice-en-es-MX"

## Part 3: set up

Finds the volis folder, switches on volis's offline settings (nothing in this notebook goes on the
internet), and checks your two language tags.

In [ ]:
import sys, csv, json, random, time, unicodedata
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from volis import paths
paths.apply_offline_environment(ROOT)       # before torch or transformers are imported

from volis import varieties, persian
from volis.translate import prompts

BASE = ROOT / "training" / "base" / MODEL.split("/")[-1]
RUN = ROOT / "training" / "runs" / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
for tag in (SOURCE, TARGET):
    varieties.require(tag)                   # stops with the list of known tags if one is wrong
print("volis folder:  ", ROOT)
print("this run:      ", RUN)
print(f"translating:    {varieties.lookup(SOURCE).prompt} -> {varieties.lookup(TARGET).prompt}")

## Part 4: get the model (once)

Training reads the model from your disk, so it has to be downloaded first, once. If the cell
below says it's missing, it prints the one command to run in PowerShell, **from the volis
folder**. Run it, wait for it to finish, then run the cell again.

Some models (Gemma, for one) ask you to accept their license on their Hugging Face page and to log
in first: `.venv\Scripts\hf auth login`.

When the model is there, the cell lists what's in the folder and what each file is for.

In [ ]:
def check_model(folder: Path) -> None:
    if not (folder / "config.json").is_file():
        print(f"The model isn't here yet: {folder}\n")
        print("Run this in PowerShell, from the volis folder, then run this cell again:\n")
        print(f'.venv\\Scripts\\hf download {MODEL} --local-dir "{folder}"')
        raise SystemExit("(stopped: no model yet)")
    files = sorted(p.name for p in folder.iterdir() if p.is_file())
    weights = [f for f in files if f.endswith(".safetensors")]
    what = {
        "config.json": "the model's blueprint: its size and shape",
        "tokenizer.json": "how text is cut into the pieces the model reads",
        "tokenizer_config.json": "tokenizer settings, often with the chat template",
        "chat_template.jinja": "the chat template: how a conversation is laid out for the model",
        "generation_config.json": "default settings for writing answers",
        "model.safetensors.index.json": "which weights are in which .safetensors part",
    }
    print(f"The model in {folder}:\n")
    for f in files:
        note = what.get(f) or ("the weights: what the model knows" if f.endswith(".safetensors") else "")
        print(f"  {f:40} {note}")
    problems = []
    if not weights:
        old = [f for f in files if f.endswith((".bin", ".pth", ".pt"))]
        packed = [f for f in files if f.endswith((".gguf", ".onnx"))]
        problems.append("no .safetensors weights. "
                        + ("It has the older PyTorch format; download the safetensors version." if old else "")
                        + ("This is a version packed for running (GGUF/ONNX), which can't be trained. "
                           "Download the original from the model's own Hugging Face page." if packed else ""))
    if not any(f in files for f in ("tokenizer.json", "tokenizer.model")):
        problems.append("no tokenizer file")
    if problems:
        raise SystemExit("Can't train this folder: " + "; ".join(problems))
    print("\nok: this is a model that can be trained.")

check_model(BASE)

## Part 5: load and check your sentence pairs

Reads your file (or the practice pairs), tidies each sentence, and drops the rows that would teach
the model something wrong. It tells you what it dropped, and shows a few of the pairs it kept.

In [ ]:
PRACTICE = [
    ("What's up, how's it going?", "¿Qué onda, cómo te va?"),
    ("That party was really cool.", "Esa fiesta estuvo bien padre."),
    ("I'll be there in a minute.", "Ahorita llego."),
    ("I have a lot of work today.", "Hoy tengo un montón de chamba."),
    ("Do you want to grab some tacos?", "¿Quieres ir por unos tacos?"),
    ("No way, that's crazy!", "¡No manches, qué locura!"),
    ("My car broke down again.", "Se me volvió a descomponer el carro."),
    ("Can you lend me some money until Friday?", "¿Me prestas una lana hasta el viernes?"),
    ("We're going to the movies tonight.", "Hoy en la noche vamos al cine."),
    ("Don't worry, it's no big deal.", "No te preocupes, no pasa nada."),
    ("The food here is really good.", "La comida aquí está bien rica."),
    ("I'm really tired, I'm going to bed.", "Estoy bien cansado, ya me voy a dormir."),
    ("My friends are coming over on Saturday.", "Mis cuates vienen el sábado."),
    ("How much does this cost?", "¿Cuánto cuesta esto?"),
    ("Hurry up, we're late!", "¡Apúrate, ya vamos tarde!"),
    ("Turn off the light, please.", "Apaga la luz, por favor."),
    ("It's really hot today.", "Hoy hace un calorón."),
    ("I didn't understand, can you say it again?", "No le entendí, ¿me lo repites?"),
    ("My mom is making dinner.", "Mi mamá está haciendo la cena."),
    ("I'll call you later.", "Al rato te marco."),
    ("The bus is always late.", "El camión siempre llega tarde."),
    ("That guy is really annoying.", "Ese güey es bien pesado."),
    ("Let's go, it's getting dark.", "Vámonos, ya se está haciendo de noche."),
    ("I need to buy juice at the store.", "Tengo que comprar un jugo en la tienda."),
    ("Where did you leave the keys?", "¿Dónde dejaste las llaves?"),
    ("We had a great time at the beach.", "Nos la pasamos súper bien en la playa."),
    ("Can you pick me up at the airport?", "¿Me puedes recoger en el aeropuerto?"),
    ("I'm broke until payday.", "Ando sin lana hasta la quincena."),
    ("This phone is really expensive.", "Este celular está bien caro."),
    ("Wait for me, I'm almost ready.", "Espérame, ya casi estoy lista."),
    ("The kids are playing outside.", "Los niños están jugando afuera."),
    ("Let me know when you get home.", "Avísame cuando llegues a tu casa."),
    ("Is there a pharmacy near here?", "¿Hay una farmacia por aquí cerca?"),
    ("I don't feel like going out today.", "Hoy no tengo ganas de salir."),
    ("That movie was really boring.", "Esa película estuvo bien aburrida."),
    ("Sure, no problem.", "Claro, sin problema."),
    ("My brother works at a restaurant.", "Mi hermano trabaja en un restaurante."),
    ("Can I pay with a card?", "¿Puedo pagar con tarjeta?"),
    ("It's raining really hard.", "Está lloviendo bien fuerte."),
    ("Bring me a cold soda, please.", "Tráeme un refresco bien frío, por favor."),
]

def read_pairs(path: Path) -> list[tuple[str, str]]:
    if path.suffix.lower() == ".jsonl":
        rows = [json.loads(line) for line in path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
        return [(r.get("source", ""), r.get("target", "")) for r in rows]
    with path.open(encoding="utf-8-sig", newline="") as f:
        sample = f.read(4096); f.seek(0)
        dialect = csv.excel_tab if path.suffix.lower() == ".tsv" or "\t" in sample.splitlines()[0] else csv.excel
        rows = list(csv.reader(f, dialect))
    if rows and [c.strip().lower() for c in rows[0][:2]] == ["source", "target"]:
        rows = rows[1:]
    elif rows:
        print(f'note: the first row isn\'t "source,target", so it was read as a pair: {rows[0][:2]}')
    return [(r[0], r[1]) if len(r) >= 2 else (r[0] if r else "", "") for r in rows]

clean_persian = persian.cleaner(ROOT, True)

def tidy(text: str, tag: str) -> str:
    text = unicodedata.normalize("NFC", text or "")
    text = " ".join(text.split())                       # one space between words, none at the ends
    return clean_persian(text, tag).text                # Persian: Arabic-style letters -> Persian ones

raw = PRACTICE if not DATA_FILE else read_pairs(Path(DATA_FILE))
kept, dropped, seen = [], {}, set()
for s, t in raw:
    s, t = tidy(s, SOURCE), tidy(t, TARGET)
    reason = ("one side is empty" if not s or not t else
              "both sides are the same" if s == t else
              "one side is over 3 times longer than the other" if max(len(s), len(t)) > 3 * min(len(s), len(t)) + 10 else
              "very long (over 400 letters): split it into sentences" if max(len(s), len(t)) > 400 else
              "a duplicate" if (s, t) in seen else "")
    if reason:
        dropped[reason] = dropped.get(reason, 0) + 1
        continue
    seen.add((s, t))
    kept.append((s, t))

print(f"{'practice pairs' if not DATA_FILE else DATA_FILE}: {len(raw)} rows, {len(kept)} kept")
for reason, n in dropped.items():
    print(f"  dropped {n}: {reason}")
print("\nA few of them:")
for s, t in random.Random(1).sample(kept, min(5, len(kept))):
    print(f"  {s}\n    -> {t}")
if len(kept) < 20:
    raise SystemExit("Fewer than 20 usable pairs: too few to train on.")

## Part 6: set some pairs aside as the exam

A few pairs are kept out of training completely. After training, the model translates them, and
its answers are compared with yours. It's an exam: if the model studied the exam questions, a good
score would prove nothing.

The exam is saved in this run's folder the first time. Later runs with the same `RUN_NAME` reuse
it, so scores from different attempts are comparable.

In [ ]:
def write_csv(path, pairs):
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.writer(f); w.writerow(["source", "target"]); w.writerows(pairs)

exam_file = RUN / "exam.csv"
if exam_file.is_file():
    exam = [tuple(r) for r in list(csv.reader(exam_file.open(encoding="utf-8")))[1:]]
    print(f"Reusing this run's exam: {len(exam)} pairs from {exam_file}")
else:
    order = kept[:]
    random.Random(0).shuffle(order)
    size = min(200, max(5, len(order) // 10))       # a tenth, between 5 and 200
    exam = order[:size]
    write_csv(exam_file, exam)
    print(f"Set aside {len(exam)} pairs as the exam: {exam_file}")

exam_sources = {s for s, _ in exam}
study = [p for p in kept if p[0] not in exam_sources]
write_csv(RUN / "study.csv", study)
print(f"{len(study)} pairs to learn from")

## Part 7: show the model each pair the way volis will ask

The model learns best from examples that look exactly like the job it'll do. So each pair is
written out as **the exact request volis sends** (volis's own translation instructions, from
`prompts\default.txt`, with your sentence in it), followed by your translation as the answer.

Only the answer is learned. The request is there so the model sees the situation; it isn't asked
to memorise the instructions. Below, the part it learns is marked.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE, local_files_only=True)
if not tokenizer.chat_template:
    raise SystemExit("This model has no chat template, so volis couldn't lay out a request for it. "
                     "Use the chat (instruct, '-it') version of the model.")
prompt_file = prompts.load(paths.prompts_dir(ROOT))
THINK = {"enable_thinking": False} if "enable_thinking" in tokenizer.chat_template else {}   # Qwen3: no "thinking out loud"

def request_text(source_text: str) -> str:
    messages = [{"role": "system", "content": prompt_file.system_text(SOURCE, TARGET)},
                {"role": "user", "content": prompt_file.user_text(source_text, SOURCE, TARGET)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **THINK)

def answer_text(source_text: str, translation: str) -> str:
    """The translation as the model should write it, including the mark that says 'I'm done'."""
    request = request_text(source_text)
    messages = [{"role": "system", "content": prompt_file.system_text(SOURCE, TARGET)},
                {"role": "user", "content": prompt_file.user_text(source_text, SOURCE, TARGET)},
                {"role": "assistant", "content": translation}]
    whole = tokenizer.apply_chat_template(messages, tokenize=False, **THINK)
    if whole.startswith(request) and translation in whole[len(request):]:
        return whole[len(request):].rstrip("\n")
    return translation + (tokenizer.eos_token or "")

def encode(pairs):
    out = []
    for s, t in pairs:
        req = tokenizer(request_text(s), add_special_tokens=False)["input_ids"]
        ans = tokenizer(answer_text(s, t), add_special_tokens=False)["input_ids"]
        ids = (req + ans)[:512]
        labels = ([-100] * len(req) + ans)[:512]    # -100 means "don't learn this part"
        if any(x != -100 for x in labels):
            out.append((ids, labels))
    return out

s, t = study[0]
print("One example, exactly as the model sees it:\n")
print(request_text(s), end="")
print("[LEARNED >>>]" + answer_text(s, t) + "[<<< LEARNED]")

## Part 8: how good is it before training?

The plain model sits the exam first, so there's something to compare with afterwards.

The score is **chrF**, from 0 to 100: how much of each translation matches yours, counted in small
groups of letters, so a nearly-right word still gets partial credit. Higher is better. The number
means most as a **comparison**, before against after, on the same exam. Don't compare it across
different exams or languages.

In [ ]:
import torch
import sacrebleu
from transformers import AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
print("Running on:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "the processor (no NVIDIA GPU found: slow)")

def load_model():
    return AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE, local_files_only=True).to(DEVICE)

def translate_all(model, sentences):
    model.eval()
    out = []
    for s in sentences:
        inputs = tokenizer(request_text(s), return_tensors="pt", add_special_tokens=False).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=128, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        out.append(tokenizer.decode(ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().strip('"'))
    return out

def score(hypotheses):
    return sacrebleu.corpus_chrf(hypotheses, [[t for _, t in exam]]).score

def free_memory():
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

model = load_model()
started = time.time()
before = translate_all(model, [s for s, _ in exam])
before_score = score(before)
del model; free_memory()
print(f"Before training: chrF {before_score:.1f}   ({len(exam)} exam sentences, {time.time() - started:.0f} s)")

## Part 9: train

The settings, in plain words:

- `RANK`: how big the LoRA is. Bigger can learn more, and can also memorise more. 16 is a common
  start.
- `EPOCHS`: how many times the model reads through all your pairs. 2 or 3 is usual. Too many, and
  it memorises your sentences instead of learning from them.
- `LEARNING_RATE`: how big a step it takes each time it adjusts. `2e-4` (0.0002) is the usual
  starting point for a LoRA.
- `BATCH_SIZE`: how many pairs it looks at together. If you see *out of memory*, halve it.

While it trains, it prints two numbers each round. **Loss** is how surprised the model is by your
translation: lower means it would have written something closer to yours.
- *training loss*: on the pairs it's learning from. It should go down.
- *check loss*: on a few pairs held back from training. If this one starts going **up** while
  training loss keeps going down, the model has started memorising. The notebook keeps the LoRA
  from the round where check loss was lowest.

In [ ]:
RANK = 16
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4

from peft import LoraConfig, get_peft_model

random.seed(0); torch.manual_seed(0)
examples = encode(study)
random.Random(0).shuffle(examples)
n_check = max(2, len(examples) // 10)
check, learn = examples[:n_check], examples[n_check:]
pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

def batches(items, size):
    for i in range(0, len(items), size):
        chunk = items[i:i + size]
        longest = max(len(ids) for ids, _ in chunk)
        ids = torch.full((len(chunk), longest), pad); labels = torch.full((len(chunk), longest), -100)
        mask = torch.zeros((len(chunk), longest), dtype=torch.long)
        for n, (a, b) in enumerate(chunk):
            ids[n, :len(a)] = torch.tensor(a); labels[n, :len(b)] = torch.tensor(b); mask[n, :len(a)] = 1
        yield ids.to(DEVICE), labels.to(DEVICE), mask.to(DEVICE), len(chunk)

def loss_of(model, batch):
    ids, labels, mask, _ = batch
    with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
        return model(input_ids=ids, attention_mask=mask, labels=labels).loss

def check_loss(model):
    model.eval(); total = count = 0
    with torch.no_grad():
        for b in batches(check, BATCH_SIZE):
            total += loss_of(model, b).item() * b[3]; count += b[3]
    model.train()
    return total / count

model = load_model()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})   # saves memory
model.enable_input_require_grads()
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=RANK, lora_alpha=2 * RANK, lora_dropout=0.05,
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"The LoRA adjusts {trainable:,} numbers; the model has {total:,}. That's {100 * trainable / total:.2f}%.")
print(f"{len(learn)} pairs to learn from, {len(check)} held back for the check.\n")

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
lora_dir = RUN / "lora"
best = check_loss(model); best_round = 0
model.save_pretrained(lora_dir)
print(f"round 0 (before training): check loss {best:.3f}")
started = time.time()
try:
    for round_ in range(1, EPOCHS + 1):
        random.Random(round_).shuffle(learn)
        losses = []
        for b in batches(learn, BATCH_SIZE):
            loss = loss_of(model, b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step(); optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())
        now = check_loss(model)
        note = ""
        if now < best:
            best, best_round = now, round_
            model.save_pretrained(lora_dir); note = "  <- best so far, saved"
        print(f"round {round_}: training loss {sum(losses) / len(losses):.3f}, check loss {now:.3f}"
              f"   ({(time.time() - started) / 60:.1f} min){note}")
except torch.cuda.OutOfMemoryError:
    raise SystemExit("Out of graphics memory. Set BATCH_SIZE to half of what it is, and run this cell again.")
finally:
    del model, optimizer; free_memory()

if best_round == 0:
    print("\nThe check loss never went below where it started: training didn't help. "
          "Usually that means too few pairs, or a LEARNING_RATE that's too high.")
else:
    print(f"\nKept the LoRA from round {best_round}, in {lora_dir}")

## Part 10: did it help?

The model sits the same exam again, this time with the LoRA on, and the two attempts are shown side
by side with your translations.

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(load_model(), lora_dir)
after = translate_all(model, [s for s, _ in exam])
after_score = score(after)
del model; free_memory()

print(f"chrF before: {before_score:.1f}    after: {after_score:.1f}    change: {after_score - before_score:+.1f}\n")
for (s, t), b, a in list(zip(exam, before, after))[:10]:
    print(s)
    print("   yours:   ", t)
    print("   before:  ", b)
    print("   after:   ", a)
    print()
(RUN / "result.json").write_text(json.dumps({
    "base_model": str(BASE), "source": SOURCE, "target": TARGET, "pairs_learned": len(study),
    "exam_pairs": len(exam), "rank": RANK, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
    "chrf_before": round(before_score, 1), "chrf_after": round(after_score, 1)}, indent=2), encoding="utf-8")

if after_score > before_score + 2:
    print("Better. Part 11 puts it into volis.")
elif after_score >= before_score - 2:
    print("About the same. More pairs, or cleaner pairs, is usually what helps next.")
else:
    print("Worse. Try fewer EPOCHS or a smaller LEARNING_RATE, and check the pairs for mistakes.")
if not DATA_FILE:
    print("(This was the practice data: 40 pairs. Expect little change; it's there to show the steps.)")

## Part 11: use it in volis

There are two ways. Pick one.

### Way A, the simplest: one ready-to-run model

The LoRA is merged into a copy of the model, and the copy goes into volis's `models\mt\` folder.
volis runs it directly, through its *transformers* translator. Press **Rescan** in the window, and
it's there.

The cost: it's a full copy of the model (about 1.2 GB for the practice model, 15 GB or more for a
big one), and this kind of model runs a little slower in volis than a GGUF one. This cell only
runs if you set `MAKE_COPY = True`.

In [ ]:
MAKE_COPY = False
COPY_NAME = RUN_NAME          # the folder name in models\mt\

if MAKE_COPY:
    dest = paths.mt_dir(ROOT) / COPY_NAME
    if dest.exists():
        raise SystemExit(f"{dest} already exists. Choose another COPY_NAME, or remove that folder first.")
    merged = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16, local_files_only=True), lora_dir
    ).merge_and_unload()
    merged.save_pretrained(dest)
    tokenizer.save_pretrained(dest)
    (dest / "volis-python.toml").write_text(f'name = "{BASE.name} + {RUN_NAME}"\n', encoding="utf-8")
    del merged; free_memory()
    print("Made", dest)
    print("In volis, press Rescan: it's in the translator list as", f'"{BASE.name} + {RUN_NAME}".')
else:
    print("Set MAKE_COPY = True and run this cell again to make the copy.")

### Way B, smaller and faster: the LoRA as a GGUF file

volis's usual translators are GGUF files. This way turns the LoRA into a small GGUF of its own,
which goes on top of the **GGUF version of the same model** (`attach-a-lora.ipynb` shows how).

It takes two commands, which run in model-converter's training environment, the one that has the
converting tools. This cell prints them; it doesn't run them. Run each in PowerShell, one at a
time.

1. **Only if volis doesn't already have this model as a GGUF:** turn the model itself into one.
   (For the practice model, it doesn't. The GGUF versions in `models\mt\` are of other models.)
2. Turn the LoRA into a GGUF.

Then follow `attach-a-lora.ipynb`, Step 3c, with the LoRA file and `base` set to the model's GGUF.

If the converter says it doesn't know the model, the copy of llama.cpp in model-converter is
older than that model; Way A still works.

In [ ]:
CONVERTERS = Path(r"D:\AI_Data\projects\model-converter")
llama = CONVERTERS / "qwen-to-gguf" / "llama.cpp"
env = CONVERTERS / "training"
base_gguf_dir = paths.mt_dir(ROOT) / BASE.name
lora_gguf = RUN / f"{RUN_NAME}-lora.gguf"

print("1. Only if there's no GGUF of this model in models\\mt yet:\n")
print(f'New-Item -ItemType Directory -Force "{base_gguf_dir}"')
print(f'uv run --project "{env}" python "{llama / "convert_hf_to_gguf.py"}" "{BASE}" --outfile "{base_gguf_dir / (BASE.name + "-F16.gguf")}" --outtype f16')
print("\n2. The LoRA:\n")
print(f'uv run --project "{env}" python "{llama / "convert_lora_to_gguf.py"}" "{lora_dir}" --base "{BASE}" --outfile "{lora_gguf}" --outtype f16')
print(f"\nThen, in attach-a-lora.ipynb: ADAPTER_FILE = r\"{lora_gguf}\", and base = \"{BASE.name}/{BASE.name}-F16.gguf\"")

## What next

- **Your own data:** set `DATA_FILE` in Part 2, give the run a new `RUN_NAME`, and run everything
  again from Part 3.
- **Trying different settings on the same data:** keep the same `RUN_NAME`, so the exam stays the
  same and the scores compare fairly. Change one setting at a time in Part 9, and run from Part 9.
- **A bigger model:** download its original version (Part 4), set `MODEL` to it, and use a
  card with enough memory (Part 1). Use the chat version of the model (`-it` or `-Instruct` in its
  name), and pick one volis also has, or can have, as a GGUF.
- **Long runs with lots of data:** model-converter's `train.ps1` runs these same steps as a job
  that can be stopped and carried on, on another machine if needed.

Everything this notebook made is in `training\runs\<RUN_NAME>\`: the exam, the pairs it learned
from, the LoRA, and the scores. Deleting that folder removes all of it.